# 03 — Evaluate trajectories and record a review

Generate measured results for 20 designed cases, inspect errors, and make a separate human decision. The cases assess this synthetic runtime; they are not a general agent benchmark or an LLM security evaluation.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, sys, urllib.request

# Replaced during publication with the commit containing the reviewed helper.
REF = "faf63f0f09cba07291e8c3d09ef95974b8404d77"
candidates = [Path("agent_studio.py"), Path("../agent_studio.py")]
source_path = next((p.resolve() for p in candidates if p.is_file()), None)
origin = "local copy"
if source_path is None:
    origin = f"GitHub commit {REF}"
    url = ("https://raw.githubusercontent.com/ZagElSayed/"
           f"Introduction-to-Neuroinformatics/{REF}/"
           "chapters/ch19-ai-agents/agent_studio.py")
    payload = urllib.request.urlopen(url, timeout=30).read()
    source_path = Path("agent_studio.py").resolve()
    source_path.write_bytes(payload)
manifest = {"origin": origin, "source_path": str(source_path),
            "sha256": hashlib.sha256(source_path.read_bytes()).hexdigest()}
Path("agent_source_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
spec = importlib.util.spec_from_file_location("agent_studio", source_path)
studio_module = importlib.util.module_from_spec(spec)
sys.modules["agent_studio"] = studio_module
spec.loader.exec_module(studio_module)
Studio = studio_module.Studio

In [ ]:
rows = studio_module.evaluate_cases("notebook03_evaluation")
for row in rows:
    print(row["case"], row["expected"], row["actual"],
          row["matches_expected"], round(row["elapsed_ms"], 2))
Path("notebook03_evaluation_table.json").write_text(json.dumps(rows, indent=2))
print("Cases:", len(rows))
print("Matching expectations:", sum(r["matches_expected"] for r in rows))

## Investigate mismatches and denominators

Open at least one successful log and one blocked/stopped log. Separate ordinary completion, correct stops, blocked proposals, and failures. A final task status does not prove that every intermediate effect was appropriate. The malicious annotation is ignored by the baseline; a real LLM's response to it remains untested. Timings include local teaching-runtime work and vary by machine.

In [ ]:
mismatches = [r for r in rows if not r["matches_expected"]]
print(json.dumps(mismatches, indent=2))
for label in ("normal", "delete_tool", "missing_units"):
    row = next(r for r in rows if r["case"] == label)
    print(label, Path(row["log"]).read_text())

## Inspect a new staged artifact

Read the report and confirm the hash. Review is a separate action, and the original report remains pending as an immutable snapshot. The review record expresses your later decision. No report is promoted to final derivatives.

In [ ]:
studio = Studio("notebook03_review")
run = studio.run("toy_good")
receipt = run["values"].get("stage_report")
if receipt:
    print(Path(receipt["path"]).read_text())
    print("Report SHA-256:", receipt["sha256"])

## Enter your own review only after inspection

The following cell does nothing until you fill in all three fields. A typed name is an educational record, not authentication. For `correct`, put the proposed correction and its rationale in the text; this helper does not silently replace a scientific value.

In [ ]:
reviewer = ""
decision = ""  # confirm, correct, or defer
rationale = ""
if reviewer and decision and rationale and receipt:
    review_receipt = studio.record_review(
        receipt, confirmed_sha256=receipt["sha256"],
        reviewer=reviewer, decision=decision, rationale=rationale)
    print(review_receipt)
else:
    print("No review recorded. Inspect the report, then enter your decision.")

## Deliver your evidence

Submit the actual table, source manifest, staged artifact, trajectory, review record, and agent card. Describe one useful next extension and a comparison that would show whether it helps. Do not interpret a high match count on these fixtures as evidence of clinical validity or general LLM reliability.